# ChestSight: Multi-Label Pathology Detection from Chest X-Rays

**Author:** Amit Tiwari  
**Institution:** University of Salford — MSc Data Science (2025)  
**Dataset:** NIH ChestX-ray14 ([Kaggle](https://www.kaggle.com/datasets/nih-chest-xrays/data))  
**Architecture:** DenseNet-121 with ImageNet pre-training  
**Framework:** TensorFlow / Keras

---

## Overview

This notebook implements a complete end-to-end pipeline for automated detection of 14 thoracic pathologies from frontal chest X-rays using a fine-tuned DenseNet-121 backbone. The pipeline covers:

1. Configuration & environment setup  
2. Data loading and label encoding  
3. Exploratory Data Analysis (EDA)  
4. Image preprocessing and augmentation  
5. Model construction with transfer learning  
6. Training with callbacks  
7. Evaluation (Classification report, IoU, Dice)  
8. Explainability via Grad-CAM  

> **Dataset note:** Download the NIH ChestX-ray14 dataset from Kaggle and place it under the path configured in Section 1. The images and `Data_Entry_2017_v2020.csv` file must be present before running cells beyond Section 2.


## 1. Configuration & Imports

In [ ]:
# ── Standard library ──────────────────────────────────────────────────────────
import os
import random
import warnings
warnings.filterwarnings('ignore')

# ── Scientific computing ───────────────────────────────────────────────────────
import numpy as np
import pandas as pd

# ── Deep learning ──────────────────────────────────────────────────────────────
import tensorflow as tf
from tensorflow.keras.applications import DenseNet121
from tensorflow.keras import layers, models, optimizers, callbacks
from tensorflow.keras.layers import GlobalAveragePooling2D, Dropout, Dense
from tensorflow.keras.models import Model

# ── Image processing ───────────────────────────────────────────────────────────
import cv2

# ── Visualisation ──────────────────────────────────────────────────────────────
import matplotlib.pyplot as plt
import seaborn as sns

# ── Metrics ────────────────────────────────────────────────────────────────────
from sklearn.metrics import classification_report, jaccard_score, f1_score

# ── Reproducibility ────────────────────────────────────────────────────────────
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
tf.random.set_seed(SEED)

print(f"TensorFlow version : {tf.__version__}")
print(f"GPU available      : {len(tf.config.list_physical_devices('GPU')) > 0}")


### 1.1 Project Configuration

All paths and hyperparameters are declared here. Update `DATA_DIR` to point to your local dataset directory before running the rest of the notebook.


In [ ]:
# ── Paths ──────────────────────────────────────────────────────────────────────
# Update this to the directory containing your NIH ChestX-ray14 files.
# Expected structure:
#   DATA_DIR/
#     Data_Entry_2017_v2020.csv
#     images/
#       00000001_000.png
#       00000001_001.png
#       ...

DATA_DIR   = "data"                                         # <-- update this path
CSV_PATH   = os.path.join(DATA_DIR, "Data_Entry_2017_v2020.csv")
IMAGE_DIR  = os.path.join(DATA_DIR, "images")
MODEL_PATH = "checkpoints/chestsight_best.weights.h5"

os.makedirs("checkpoints", exist_ok=True)
os.makedirs("outputs",     exist_ok=True)

# ── Image settings ─────────────────────────────────────────────────────────────
IMG_SIZE   = 224          # DenseNet-121 expected input size
CHANNELS   = 3

# ── Training hyperparameters ───────────────────────────────────────────────────
BATCH_SIZE    = 16
EPOCHS        = 10
LEARNING_RATE = 1e-4
DROPOUT_RATE  = 0.5

# ── Data split ratios ──────────────────────────────────────────────────────────
TRAIN_RATIO = 0.70
VAL_RATIO   = 0.10
TEST_RATIO  = 0.20

# ── ImageNet normalisation constants ──────────────────────────────────────────
IMAGENET_MEAN = [0.485, 0.456, 0.406]
IMAGENET_STD  = [0.229, 0.224, 0.225]

print("Configuration loaded successfully.")
print(f"  Data directory : {DATA_DIR}")
print(f"  Image size     : {IMG_SIZE}x{IMG_SIZE}x{CHANNELS}")
print(f"  Batch size     : {BATCH_SIZE}")
print(f"  Epochs         : {EPOCHS}")


## 2. Data Loading & Label Encoding

In [ ]:
# ── Load metadata CSV ──────────────────────────────────────────────────────────
df = pd.read_csv(CSV_PATH)
print(f"Dataset loaded: {len(df):,} records | {df.shape[1]} columns")
print(f"Columns: {df.columns.tolist()}")
df.head()


In [ ]:
# ── Extract unique labels ──────────────────────────────────────────────────────
all_labels = set()
for label_str in df['Finding Labels']:
    for label in label_str.split('|'):
        all_labels.add(label.strip())

all_labels = sorted(list(all_labels))
label2idx  = {label: idx for idx, label in enumerate(all_labels)}
num_labels = len(all_labels)

print(f"Number of unique labels: {num_labels}")
print(f"Labels: {all_labels}")


# ── Multi-hot encoding ─────────────────────────────────────────────────────────
def encode_labels(label_str: str) -> np.ndarray:
    """Convert a pipe-delimited label string to a multi-hot binary vector."""
    label_vector = np.zeros(num_labels, dtype=np.float32)
    for label in label_str.split('|'):
        label = label.strip()
        if label in label2idx:
            label_vector[label2idx[label]] = 1.0
    return label_vector

df['encoded'] = df['Finding Labels'].apply(encode_labels)
print(f"\nLabel encoding complete. Shape of first vector: {df['encoded'].iloc[0].shape}")


## 3. Exploratory Data Analysis (EDA)

### 3.1 Dataset Overview


In [ ]:
print("── DataFrame Info ──────────────────────────────────────────────")
print(df.info())
print("\n── Missing Values ──────────────────────────────────────────────")
print(df.isnull().sum())
print("\n── Numerical Summary ───────────────────────────────────────────")
print(df.describe())


### 3.2 Label Distribution

In [ ]:
# ── Per-label frequencies ──────────────────────────────────────────────────────
label_frequencies = {}
for label_str in df['Finding Labels']:
    for label in label_str.split('|'):
        label = label.strip()
        label_frequencies[label] = label_frequencies.get(label, 0) + 1

total_images = len(df)
label_df = (
    pd.DataFrame.from_dict(label_frequencies, orient='index', columns=['Count'])
    .assign(Percentage=lambda x: (x['Count'] / total_images * 100).round(2))
    .sort_values('Count', ascending=False)
)
print(label_df.to_string())

# ── Bar chart ──────────────────────────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(14, 5))
ax.bar(label_df.index, label_df['Count'], color='steelblue', edgecolor='white')
ax.set_xlabel("Pathology", fontsize=12)
ax.set_ylabel("Count", fontsize=12)
ax.set_title("Label Frequency Distribution — NIH ChestX-ray14", fontsize=14, fontweight='bold')
ax.tick_params(axis='x', rotation=45)
sns.despine()
plt.tight_layout()
plt.savefig("outputs/label_distribution.png", dpi=150, bbox_inches='tight')
plt.show()


### 3.3 Multi-label Co-occurrence

In [ ]:
# ── Labels per image distribution ─────────────────────────────────────────────
label_counts_per_image = df['Finding Labels'].str.split('|').apply(len)

single   = (label_counts_per_image == 1).sum()
double   = (label_counts_per_image == 2).sum()
triple   = (label_counts_per_image == 3).sum()
quad_plus= (label_counts_per_image >= 4).sum()

print("Labels per image:")
print(f"  Single   : {single:>6,}  ({single/total_images*100:.2f}%)")
print(f"  Double   : {double:>6,}  ({double/total_images*100:.2f}%)")
print(f"  Triple   : {triple:>6,}  ({triple/total_images*100:.2f}%)")
print(f"  4+       : {quad_plus:>6,}  ({quad_plus/total_images*100:.2f}%)")

# ── Pie chart ──────────────────────────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(6, 6))
sizes   = [single, double, triple, quad_plus]
lbls    = ['Single', 'Double', 'Triple', '4+']
ax.pie(sizes, labels=lbls, autopct='%1.1f%%', startangle=60,
       explode=(0.05, 0, 0, 0), colors=['#4C72B0','#DD8452','#55A868','#C44E52'])
ax.set_title("Distribution of Label Counts per Image", fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig("outputs/multilabel_distribution.png", dpi=150, bbox_inches='tight')
plt.show()


### 3.4 Patient Demographics

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# Gender distribution
gender_pct = (df['Patient Gender'].value_counts() / total_images * 100)
axes[0].pie(gender_pct, labels=gender_pct.index, autopct='%1.1f%%',
            colors=['#4C72B0','#DD8452'], startangle=90)
axes[0].set_title("Gender Distribution", fontsize=12, fontweight='bold')

# View position distribution
view_pct = (df['View Position'].value_counts() / total_images * 100)
axes[1].pie(view_pct, labels=view_pct.index, autopct='%1.1f%%',
            colors=['#55A868','#C44E52'], startangle=90)
axes[1].set_title("View Position Distribution", fontsize=12, fontweight='bold')

# Age distribution
axes[2].hist(df['Patient Age'], bins=range(0, 100, 5),
             color='steelblue', edgecolor='white')
axes[2].set_xlabel("Age", fontsize=11)
axes[2].set_ylabel("Frequency", fontsize=11)
axes[2].set_title("Patient Age Distribution", fontsize=12, fontweight='bold')
sns.despine(ax=axes[2])

plt.suptitle("Patient Demographics — NIH ChestX-ray14", fontsize=14,
             fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig("outputs/demographics.png", dpi=150, bbox_inches='tight')
plt.show()


### 3.5 Label Co-occurrence Heatmap

In [ ]:
# ── Co-occurrence matrix ───────────────────────────────────────────────────────
co_occurrence = np.zeros((num_labels, num_labels))
for _, row in df.iterrows():
    lbls = [l.strip() for l in row['Finding Labels'].split('|')]
    for i in range(len(lbls)):
        for j in range(i + 1, len(lbls)):
            if lbls[i] in label2idx and lbls[j] in label2idx:
                co_occurrence[label2idx[lbls[i]], label2idx[lbls[j]]] += 1
                co_occurrence[label2idx[lbls[j]], label2idx[lbls[i]]] += 1

co_occurrence_df  = pd.DataFrame(co_occurrence, index=all_labels, columns=all_labels)
correlation_matrix = co_occurrence_df.corr()

fig, ax = plt.subplots(figsize=(14, 11))
mask = np.triu(np.ones_like(correlation_matrix, dtype=bool))
sns.heatmap(
    correlation_matrix, mask=mask, annot=True, fmt=".2f",
    cmap='coolwarm', center=0, linewidths=0.5,
    annot_kws={"size": 7}, ax=ax
)
ax.set_title("Label Co-occurrence Correlation Matrix", fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig("outputs/cooccurrence_heatmap.png", dpi=150, bbox_inches='tight')
plt.show()


### 3.6 Sample Images per Pathology

In [ ]:
# ── Display 2 sample images per class ─────────────────────────────────────────
selected = {}
for label in all_labels:
    subset = df[df['Finding Labels'].str.contains(label, regex=False)]
    if len(subset) >= 2:
        selected[label] = random.sample(list(subset['Image Index']), 2)

image_indices = [img for imgs in selected.values() for img in imgs]
num_cols = 4
num_rows = (len(image_indices) + num_cols - 1) // num_cols

fig, axes = plt.subplots(num_rows, num_cols, figsize=(20, num_rows * 5))
axes = axes.ravel()

for i, image_index in enumerate(image_indices):
    img_path = os.path.join(IMAGE_DIR, image_index)
    try:
        img = plt.imread(img_path)
        axes[i].imshow(img, cmap='gray')
        finding = df.loc[df['Image Index'] == image_index, 'Finding Labels'].iloc[0]
        axes[i].set_title(finding, fontsize=8)
        axes[i].axis('off')
    except FileNotFoundError:
        axes[i].axis('off')

# Hide unused subplots
for j in range(len(image_indices), len(axes)):
    axes[j].axis('off')

plt.suptitle("Sample Images per Pathology Class", fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig("outputs/sample_images.png", dpi=100, bbox_inches='tight')
plt.show()


## 4. Data Split & Preprocessing Pipeline

In [ ]:
# ── Train / Validation / Test split ───────────────────────────────────────────
filenames = df['Image Index'].values
labels_encoded = np.stack(df['encoded'].values)

num_samples = len(filenames)
indices = np.arange(num_samples)
np.random.shuffle(indices)

train_end = int(TRAIN_RATIO * num_samples)
val_end   = train_end + int(VAL_RATIO * num_samples)

train_idx = indices[:train_end]
val_idx   = indices[train_end:val_end]
test_idx  = indices[val_end:]

train_filenames, train_labels = filenames[train_idx], labels_encoded[train_idx]
val_filenames,   val_labels   = filenames[val_idx],   labels_encoded[val_idx]
test_filenames,  test_labels  = filenames[test_idx],  labels_encoded[test_idx]

print(f"Split summary (total: {num_samples:,})")
print(f"  Train      : {len(train_idx):>6,}  ({len(train_idx)/num_samples*100:.1f}%)")
print(f"  Validation : {len(val_idx):>6,}  ({len(val_idx)/num_samples*100:.1f}%)")
print(f"  Test       : {len(test_idx):>6,}  ({len(test_idx)/num_samples*100:.1f}%)")

# ── Pie chart ──────────────────────────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(5, 5))
ax.pie(
    [len(train_idx), len(val_idx), len(test_idx)],
    labels=['Train (70%)', 'Validation (10%)', 'Test (20%)'],
    autopct='%1.1f%%', startangle=90,
    colors=['#4C72B0', '#55A868', '#DD8452']
)
ax.set_title("Data Split Distribution", fontsize=12, fontweight='bold')
plt.tight_layout()
plt.savefig("outputs/data_split.png", dpi=150, bbox_inches='tight')
plt.show()


In [ ]:
# ── Preprocessing & augmentation pipeline ─────────────────────────────────────
def preprocess_image(filename: tf.Tensor, label: tf.Tensor, training: bool = True):
    """
    Load, decode, resize, normalise and (optionally) augment a chest X-ray image.

    Parameters
    ----------
    filename : tf.Tensor (string)
        Filename of the image within IMAGE_DIR.
    label : tf.Tensor
        Multi-hot encoded label vector.
    training : bool
        If True, apply random augmentation for training.

    Returns
    -------
    img : tf.Tensor  (IMG_SIZE x IMG_SIZE x 3, float32, ImageNet-normalised)
    label : tf.Tensor
    """
    img_path = tf.strings.join([IMAGE_DIR, filename], separator=os.sep)
    img = tf.io.read_file(img_path)
    img = tf.image.decode_png(img, channels=CHANNELS)
    img = tf.image.convert_image_dtype(img, tf.float32)

    if training:
        # Resize slightly larger, then random crop to IMG_SIZE
        img = tf.image.resize(img, [int(IMG_SIZE * 1.15), int(IMG_SIZE * 1.15)])
        img = tf.image.random_crop(img, size=[IMG_SIZE, IMG_SIZE, CHANNELS])
        img = tf.image.random_flip_left_right(img)
        img = tf.image.random_brightness(img, max_delta=0.10)
        img = tf.image.random_contrast(img, lower=0.90, upper=1.10)
    else:
        img = tf.image.resize(img, [IMG_SIZE, IMG_SIZE])

    # ImageNet normalisation
    img = (img - IMAGENET_MEAN) / IMAGENET_STD
    return img, label


# ── Build tf.data pipelines ────────────────────────────────────────────────────
def build_dataset(fnames, lbls, training: bool = True) -> tf.data.Dataset:
    ds = tf.data.Dataset.from_tensor_slices((fnames, lbls))
    ds = ds.map(
        lambda f, l: preprocess_image(f, l, training=training),
        num_parallel_calls=tf.data.AUTOTUNE
    )
    if training:
        ds = ds.shuffle(buffer_size=1000, seed=SEED)
    return ds.batch(BATCH_SIZE).prefetch(tf.data.AUTOTUNE)


train_ds = build_dataset(train_filenames, train_labels, training=True)
val_ds   = build_dataset(val_filenames,   val_labels,   training=False)
test_ds  = build_dataset(test_filenames,  test_labels,  training=False)

print("tf.data pipelines ready.")
print(f"  train_ds : {len(train_filenames):,} images | {len(train_filenames)//BATCH_SIZE} batches")
print(f"  val_ds   : {len(val_filenames):,} images | {len(val_filenames)//BATCH_SIZE} batches")
print(f"  test_ds  : {len(test_filenames):,} images | {len(test_filenames)//BATCH_SIZE} batches")


In [ ]:
# ── Visualise preprocessing ────────────────────────────────────────────────────
example_filename = df['Image Index'].iloc[0]
example_label    = df['encoded'].iloc[0]

img_train, _ = preprocess_image(example_filename, example_label, training=True)
img_val,   _ = preprocess_image(example_filename, example_label, training=False)

# Read original for comparison
orig_path = tf.strings.join([IMAGE_DIR, example_filename], separator=os.sep)
orig_img  = tf.image.convert_image_dtype(
    tf.image.decode_png(tf.io.read_file(orig_path), channels=CHANNELS), tf.float32
)

def denorm(img):
    return tf.clip_by_value(img * IMAGENET_STD + IMAGENET_MEAN, 0, 1)

fig, axes = plt.subplots(1, 3, figsize=(14, 4))
axes[0].imshow(orig_img);            axes[0].set_title("Original",               fontsize=11)
axes[1].imshow(denorm(img_train));   axes[1].set_title("Augmented (train)",       fontsize=11)
axes[2].imshow(denorm(img_val));     axes[2].set_title("Resized (val/test)",      fontsize=11)
for ax in axes: ax.axis('off')
plt.suptitle("Preprocessing Comparison", fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig("outputs/preprocessing_comparison.png", dpi=150, bbox_inches='tight')
plt.show()


## 5. Model Architecture

### 5.1 DenseNet-121 with Custom Classification Head

The backbone is DenseNet-121 pre-trained on ImageNet. Layers up to `conv5_block16_concat` are frozen; the final dense block and the custom head are left trainable, following a progressive fine-tuning strategy.


In [ ]:
# ── Build model ────────────────────────────────────────────────────────────────
FINETUNE_FROM = "conv5_block16_concat"   # Unfreeze from this layer onwards

base_model = DenseNet121(
    weights='imagenet',
    include_top=False,
    input_shape=(IMG_SIZE, IMG_SIZE, CHANNELS)
)

feature_extractor = Model(
    inputs=base_model.input,
    outputs=base_model.get_layer(FINETUNE_FROM).output,
    name="densenet121_backbone"
)

# ── Custom classification head ─────────────────────────────────────────────────
x       = feature_extractor.output
x       = GlobalAveragePooling2D(name="gap")(x)
x       = Dropout(DROPOUT_RATE, name="dropout")(x)
outputs = Dense(num_labels, activation='sigmoid', name="predictions")(x)

model = Model(inputs=base_model.input, outputs=outputs, name="ChestSight")

# ── Selective layer freezing ───────────────────────────────────────────────────
freeze = True
for layer in model.layers:
    if layer.name == FINETUNE_FROM:
        freeze = False
    layer.trainable = not freeze

trainable_count = sum(1 for l in model.layers if l.trainable)
print(f"Total layers    : {len(model.layers)}")
print(f"Trainable layers: {trainable_count}")
print(f"Frozen layers   : {len(model.layers) - trainable_count}")


In [ ]:
# ── Compile ────────────────────────────────────────────────────────────────────
model.compile(
    optimizer=optimizers.Adam(learning_rate=LEARNING_RATE),
    loss='binary_crossentropy',
    metrics=['accuracy']
)

model.summary()


## 6. Training

In [ ]:
# ── Callbacks ──────────────────────────────────────────────────────────────────
lr_scheduler = callbacks.ReduceLROnPlateau(
    monitor='val_loss', factor=0.1, patience=3, verbose=1, min_lr=1e-7
)
checkpoint = callbacks.ModelCheckpoint(
    MODEL_PATH, monitor='val_loss', save_best_only=True,
    save_weights_only=True, verbose=1
)
early_stop = callbacks.EarlyStopping(
    monitor='val_loss', patience=5, restore_best_weights=True, verbose=1
)

# ── Train ───────────────────────────────────────────────────────────────────────
history = model.fit(
    train_ds,
    epochs=EPOCHS,
    validation_data=val_ds,
    callbacks=[lr_scheduler, checkpoint, early_stop]
)


In [ ]:
# ── Training curves ────────────────────────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(14, 4))

axes[0].plot(history.history['loss'],     label='Train loss',      color='#4C72B0')
axes[0].plot(history.history['val_loss'], label='Validation loss', color='#DD8452')
axes[0].set_xlabel("Epoch"); axes[0].set_ylabel("Loss")
axes[0].set_title("Training vs Validation Loss", fontweight='bold')
axes[0].legend(); sns.despine(ax=axes[0])

axes[1].plot(history.history['accuracy'],     label='Train accuracy',      color='#4C72B0')
axes[1].plot(history.history['val_accuracy'], label='Validation accuracy', color='#DD8452')
axes[1].set_xlabel("Epoch"); axes[1].set_ylabel("Accuracy")
axes[1].set_title("Training vs Validation Accuracy", fontweight='bold')
axes[1].legend(); sns.despine(ax=axes[1])

plt.tight_layout()
plt.savefig("outputs/training_curves.png", dpi=150, bbox_inches='tight')
plt.show()


## 7. Evaluation

In [ ]:
# ── Collect predictions on validation set ─────────────────────────────────────
y_true_list, y_pred_list = [], []

for images, lbls in val_ds:
    preds = model.predict(images, verbose=0)
    preds_binary = (preds > 0.5).astype(int)
    y_true_list.extend(lbls.numpy())
    y_pred_list.extend(preds_binary)

y_true = np.array(y_true_list)
y_pred = np.array(y_pred_list)

print(f"Evaluation set  : {len(y_true):,} samples")
print(f"y_true shape    : {y_true.shape}")
print(f"y_pred shape    : {y_pred.shape}")


### 7.1 Classification Report

In [ ]:
print(classification_report(y_true, y_pred, target_names=all_labels, zero_division=0))


### 7.2 Per-class IoU and Dice Coefficient

In [ ]:
def calculate_metrics(y_true: np.ndarray, y_pred: np.ndarray, labels: list) -> pd.DataFrame:
    """
    Calculate per-class IoU (Jaccard), Dice (F1), Precision, and Recall.

    Parameters
    ----------
    y_true : np.ndarray  shape (N, num_classes)
    y_pred : np.ndarray  shape (N, num_classes)
    labels : list of str

    Returns
    -------
    pd.DataFrame  indexed by label with columns [IoU, Dice, Precision, Recall]
    """
    rows = []
    for i, label in enumerate(labels):
        tp = np.sum((y_true[:, i] == 1) & (y_pred[:, i] == 1))
        fp = np.sum((y_true[:, i] == 0) & (y_pred[:, i] == 1))
        fn = np.sum((y_true[:, i] == 1) & (y_pred[:, i] == 0))

        precision = tp / (tp + fp) if (tp + fp) > 0 else 0.0
        recall    = tp / (tp + fn) if (tp + fn) > 0 else 0.0
        iou       = jaccard_score(y_true[:, i], y_pred[:, i], zero_division=0)
        dice      = f1_score(y_true[:, i], y_pred[:, i], zero_division=0)

        rows.append({
            'Label'    : label,
            'IoU'      : round(iou, 4),
            'Dice'     : round(dice, 4),
            'Precision': round(precision, 4),
            'Recall'   : round(recall, 4),
        })

    return pd.DataFrame(rows).set_index('Label')


metrics_df = calculate_metrics(y_true, y_pred, all_labels)
print(metrics_df.to_string())

print("\n── Aggregate Metrics ──────────────────────────────────────────────────────")
print(f"  Mean IoU       : {metrics_df['IoU'].mean():.4f}")
print(f"  Mean Dice      : {metrics_df['Dice'].mean():.4f}")
print(f"  Mean Precision : {metrics_df['Precision'].mean():.4f}")
print(f"  Mean Recall    : {metrics_df['Recall'].mean():.4f}")

metrics_df.to_csv("outputs/per_class_metrics.csv")


### 7.3 Label Correlation in Predictions

In [ ]:
# ── Correlation matrix of predicted labels ─────────────────────────────────────
pred_df  = pd.DataFrame(y_true, columns=all_labels)
corr_mat = pred_df.corr()

fig, ax = plt.subplots(figsize=(14, 11))
sns.heatmap(
    corr_mat, annot=True, fmt=".2f", cmap='coolwarm',
    center=0, linewidths=0.5, annot_kws={"size": 7}, ax=ax
)
ax.set_title("Correlation Matrix — True Labels (Validation Set)",
             fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig("outputs/label_correlation_matrix.png", dpi=150, bbox_inches='tight')
plt.show()


## 8. Explainability — Grad-CAM

Grad-CAM (Gradient-weighted Class Activation Mapping) highlights which regions of the input image most influenced the model's prediction for a given class. This is critical for clinical trust and qualitative validation against anatomical knowledge.


In [ ]:
def get_gradcam_heatmap(
    img_array: tf.Tensor,
    model: tf.keras.Model,
    last_conv_layer_name: str,
    pred_index: int = None
) -> np.ndarray:
    """
    Compute a Grad-CAM heatmap for the given image and target class.

    Parameters
    ----------
    img_array : tf.Tensor  shape (1, H, W, C) — preprocessed image batch
    model : tf.keras.Model
    last_conv_layer_name : str  — name of the target convolutional layer
    pred_index : int or None    — target class index; defaults to argmax prediction

    Returns
    -------
    heatmap : np.ndarray  shape (h, w) in [0, 1]
    """
    conv_layer = model.get_layer(last_conv_layer_name)
    grad_model = tf.keras.models.Model(
        inputs=model.inputs,
        outputs=[conv_layer.output, model.output]
    )

    with tf.GradientTape() as tape:
        conv_outputs, predictions = grad_model(img_array)
        if pred_index is None:
            pred_index = tf.argmax(predictions[0])
        target = predictions[:, pred_index]

    grads       = tape.gradient(target, conv_outputs)
    pooled_grads = tf.reduce_mean(grads, axis=(0, 1, 2))

    heatmap = conv_outputs[0] @ pooled_grads[..., tf.newaxis]
    heatmap = tf.squeeze(heatmap)
    heatmap = tf.maximum(heatmap, 0) / (tf.reduce_max(heatmap) + 1e-8)
    return heatmap.numpy()


def display_gradcam(
    img_path: str,
    heatmap: np.ndarray,
    predicted_class: str,
    alpha: float = 0.4,
    threshold: float = 0.5
) -> None:
    """
    Display original image, Grad-CAM heatmap, overlay, and pseudo-segmentation mask.
    """
    orig_img      = cv2.cvtColor(cv2.imread(img_path), cv2.COLOR_BGR2RGB)
    heatmap_sized = cv2.resize(heatmap, (orig_img.shape[1], orig_img.shape[0]))
    heatmap_color = cv2.cvtColor(
        cv2.applyColorMap(np.uint8(255 * heatmap_sized), cv2.COLORMAP_JET),
        cv2.COLOR_BGR2RGB
    )
    overlay   = np.uint8(heatmap_color * alpha + orig_img)
    pseudo_mask = (heatmap_sized > threshold).astype(np.uint8) * 255

    fig, axes = plt.subplots(1, 4, figsize=(18, 4))
    panels = [
        (orig_img,      "Original image",         'gray'),
        (heatmap_sized, "Grad-CAM heatmap",        'jet'),
        (overlay,       "Overlay",                 None),
        (pseudo_mask,   "Pseudo-segmentation mask",'gray'),
    ]
    for ax, (img, title, cmap) in zip(axes, panels):
        ax.imshow(img, cmap=cmap)
        ax.set_title(title, fontsize=10, fontweight='bold')
        ax.axis('off')

    plt.suptitle(f"Grad-CAM — Predicted class: {predicted_class}",
                 fontsize=12, fontweight='bold')
    plt.tight_layout()
    plt.savefig(f"outputs/gradcam_{predicted_class.replace(' ', '_')}.png",
                dpi=150, bbox_inches='tight')
    plt.show()


In [ ]:
# ── Run Grad-CAM on a sample image ────────────────────────────────────────────
# Select the first available image from the test set
sample_filename = test_filenames[0]
sample_img_path = os.path.join(IMAGE_DIR, sample_filename)

# Preprocess for inference
img_array = load_preprocess_image(sample_img_path)

# Get prediction and top class
raw_preds = model.predict(img_array, verbose=0)[0]
top_class_idx  = int(np.argmax(raw_preds))
top_class_name = all_labels[top_class_idx]
top_class_prob = raw_preds[top_class_idx]

print(f"Sample image    : {sample_filename}")
print(f"Top prediction  : {top_class_name}  (p = {top_class_prob:.3f})")

# Compute heatmap
heatmap = get_gradcam_heatmap(img_array, model, FINETUNE_FROM, pred_index=top_class_idx)

# Visualise
display_gradcam(sample_img_path, heatmap, predicted_class=top_class_name)


def load_preprocess_image(img_path: str, target_size: tuple = (IMG_SIZE, IMG_SIZE)) -> tf.Tensor:
    """Load and preprocess a single image for inference (returns batch of 1)."""
    img = tf.io.read_file(img_path)
    img = tf.image.decode_png(img, channels=CHANNELS)
    img = tf.image.convert_image_dtype(img, tf.float32)
    img = tf.image.resize(img, target_size)
    img = (img - IMAGENET_MEAN) / IMAGENET_STD
    return tf.expand_dims(img, axis=0)


## 9. Results Summary & Next Steps

### Results

| Metric            | Value  |
|-------------------|--------|
| Micro-avg F1      | 0.49   |
| Macro-avg F1      | 0.05   |
| No Finding F1     | 0.72   |
| Mean IoU          | —      |
| Mean Dice         | —      |

> Scores above are from the baseline run. Run `calculate_metrics(y_true, y_pred, all_labels)` after training to populate the table.

### Known Limitation

The low recall on pathological classes is a direct consequence of class imbalance (57% "No Finding") combined with a default sigmoid threshold of 0.5 and Binary Cross-Entropy loss. This is the expected baseline outcome.

### Recommended Next Steps

1. Replace BCE loss with **Focal Loss** to address class imbalance at training time.  
2. Apply **per-class threshold optimisation** on the validation set to maximise per-class F1.  
3. Re-split at **patient level** to eliminate data leakage across train/val/test.  
4. Report **per-class AUC-ROC** (the standard NIH benchmark metric) alongside F1.  
5. Add **Grad-CAM++** (sharper localisation than vanilla Grad-CAM) for improved explainability.

### References

- Wang et al. (2017). *ChestX-ray8: Hospital-Scale Chest X-ray Database and Benchmarks.* CVPR.  
- Rajpurkar et al. (2017). *CheXNet: Radiologist-Level Pneumonia Detection on Chest X-Rays with Deep Learning.* arXiv:1711.05225.  
- Huang et al. (2017). *Densely Connected Convolutional Networks.* CVPR.  
- Selvaraju et al. (2017). *Grad-CAM: Visual Explanations from Deep Networks via Gradient-Based Localization.* ICCV.  
- Lin et al. (2017). *Focal Loss for Dense Object Detection.* ICCV.
